In [76]:
import pandas as pd
import numpy as np
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt

from sklearn.metrics import r2_score, mean_squared_error


from sklearn.linear_model import LinearRegression
from sklearn.model_selection import train_test_split

from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA

from scipy.stats import spearmanr

X_df =pd.read_csv('C:/Users/Julien GILLES/Documents/ESILV A4 Machine learning Projet/X_train_NHkHMNU.csv', delimiter= ',')
y_df =pd.read_csv('C:/Users/Julien GILLES/Documents/ESILV A4 Machine learning Projet/y_train_ZAN5mwg.csv', delimiter= ',')
X_test_df =pd.read_csv('C:/Users/Julien GILLES/Documents/ESILV A4 Machine learning Projet/X_test_final.csv', delimiter= ',')
df = pd.merge(X_df,y_df,on='ID')

In [79]:
def Model_Final(df,model_fr, model_de):
    #1. 
    #Split the original dataset in 2 datasets : one for the rows with COUNTRY = FR and one for the rows with COUNTRY = DE
    # For FR :
    df_fr = df[df['COUNTRY'] == 'FR']
    # For DE :
    df_de = df[df['COUNTRY'] == 'DE']

    #2.
    #Elimination des colonnes qui ne concernent pas directement le pays
    columns_to_drop_fr = ['COUNTRY','DE_CONSUMPTION', 'DE_FR_EXCHANGE', 'FR_NET_EXPORT','DE_NET_EXPORT','DE_NET_IMPORT','DE_GAS','DE_COAL','DE_HYDRO','DE_NUCLEAR','DE_SOLAR','DE_WINDPOW','DE_LIGNITE','DE_RESIDUAL_LOAD','DE_RAIN','DE_WIND','DE_TEMP']
    df_fr = df_fr.drop(columns=columns_to_drop_fr)
    columns_to_drop_de = ['COUNTRY','FR_CONSUMPTION', 'FR_DE_EXCHANGE','DE_NET_EXPORT', 'FR_NET_EXPORT','FR_NET_IMPORT','FR_GAS','FR_COAL','FR_HYDRO','FR_NUCLEAR','FR_SOLAR','FR_WINDPOW', 'FR_RESIDUAL_LOAD','FR_RAIN','FR_WIND','FR_TEMP']
    df_de = df_de.drop(columns=columns_to_drop_de)

    #3.
    #Remplacement des valeurs manquantes par la moyenne de leurs colonnes dans chaque dataset 
    df_fr, df_de = [df.fillna(df.mean()) for df in [df_fr, df_de]]

    #5.
    #Implémentation du modèle pour FR
    X_fr = df_fr.drop(columns=['ID', 'TARGET'])
    y_fr = df_fr['TARGET']
    
    X_train_fr, X_test_fr, y_train_fr, y_test_fr = train_test_split(X_fr, y_fr, test_size=0.2, random_state=42)
    #standardisation :
    scaler = StandardScaler()
    X_train_scaled_fr = scaler.fit_transform(X_train_fr)
    X_test_scaled_fr = scaler.transform(X_test_fr)

    model_fr.fit(X_train_scaled_fr, y_train_fr)

    y_pred_fr= model_fr.predict(X_test_scaled_fr)

    #6.
    #Implémentation du modèles pour DE
    X_de = df_de.drop(columns=['ID', 'TARGET'])
    y_de = df_de['TARGET']
    
    X_train_de, X_test_de, y_train_de, y_test_de = train_test_split(X_de, y_de, test_size=0.2, random_state=42)
    #standardisation :
    scaler = StandardScaler()
    X_train_scaled_de = scaler.fit_transform(X_train_de)
    X_test_scaled_de = scaler.transform(X_test_de)

    model_de.fit(X_train_scaled_de, y_train_de)

    y_pred_de = model_de.predict(X_test_scaled_de)

    #7.
    #Analyse des résultats
    #pour FR :
    
    #score R2 - FR
    score_fr = model.score(X_test_scaled_fr, y_test_fr)
    print(f"R2 score FR: {score_fr}")

    #score Spearman - FR
    spearman = spearmanr(y_test_fr, y_pred_fr)
    print(f"Spearman score FR: {spearman.correlation}")

    #MSE - FR
    mse_fr = mean_squared_error(y_test_fr, y_pred_fr)
    print(f"MSE FR: {mse_fr}")
    
    #pour DE :
    
    #score R2 - DE
    score_de = model_de.score(X_test_scaled_de, y_test_de)
    print(f"R2 score DE: {score_de}")

    #score Spearman - DE
    spearman = spearmanr(y_test_de, y_pred_de)
    print(f"Spearman score DE: {spearman.correlation}")

    #MSE - DE
    mse_de = mean_squared_error(y_test_de, y_pred_de)
    print(f"MSE DE: {mse_de}")

    #8.
    # Fusionner les résultats
    # Concaténer les prédictions et les vrais résultats pour FR et DE
    y_true_combined = pd.concat([y_test_fr, y_test_de], axis=0)
    y_pred_combined = pd.concat([pd.Series(y_pred_fr), pd.Series(y_pred_de)], axis=0)

    # Calculer les scores pour la combinaison
    combined_r2 = r2_score(y_true_combined, y_pred_combined)
    print(f"Combined R2 score: {combined_r2}")

    combined_spearman = spearmanr(y_true_combined, y_pred_combined)
    print(f"Combined Spearman score: {combined_spearman.correlation}")

    combined_mse = mean_squared_error(y_true_combined, y_pred_combined)
    print(f"Combined MSE: {combined_mse}")



In [80]:
Model_Final(df,LinearRegression(),LinearRegression())

R2 score FR: 0.001308843996675746
Spearman score FR: 0.12196349747014068
MSE FR: 1.4081039065999932
R2 score DE: 0.05029220088485209
Spearman score DE: 0.4148423524150269
MSE DE: 0.9050128222656999
Combined R2 score: 0.019226057856309597
Combined Spearman score: 0.2760418564837022
Combined MSE: 1.191774740336247
